# Exercise: Pipelines

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

import sys
import os
sys.path.append(os.path.abspath('../..')) # add parent directory to search path for modules

from src.utilities import read_filter_data

# Read data, filter rows with missing target, separate target from predictors
X_full, X_test_full, y = read_filter_data()

# Break off validation set from training data
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X_full, y, 
                                                                train_size=0.8, test_size=0.2,
                                                                random_state=0)

# Select categorical columns with relatively low cardinality (convenient but arbitrary)
categorical_cols = [ # loops through columns, appends categorical columns that have < 10 unique values to list
    cname for cname in X_train_full.columns
    if X_train_full[cname].nunique() < 10
    and X_train_full[cname].dtype == "string"
]

# Select numerical columns
numerical_cols = [ # loops through columns, appends numerical columns to list
    cname for cname in X_train_full.columns
    if X_train_full[cname].dtype in ['int64', 'float64']
]

my_cols = categorical_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid =  X_valid_full[my_cols].copy()
X_test = X_test_full[my_cols].copy()

In [ ]:
X_train.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='constant')

# Preprocessing for categorical data
categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Define model
model = RandomForestRegressor(n_estimators=100, random_state=0)

# Bundle preprocessing and modeling code in a pipeline
clf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)

# Preprocessing of training data, fit model
clf.fit(X_train, y_train)

# Preprocessing of validation data, get predications
preds = clf.predict(X_valid)

print('MAE:', mean_absolute_error(y_valid, preds))

The code cell below allows you to view each column in categorical_cols with their respective category counts (value_counts()) and number of unique values (nunique()).

This helps inspect the distribution of categories in each feature and identify columns with high cardinality or rare categories. Understaind these distributions is useful when deciding how to encode categorical variables (for example, determining whether techniques like one-hot encoding with min_frequency or dropping high-cardinality columns may be appropriate, if that wasn't filtered out already)

In [ ]:
for col in categorical_cols:
    print(X_train[col].value_counts())
    print(f"Unique values: {X_train[col].nunique()}")
    print()

Print column with the most unique values. Since we already filtered out columns that had more than 10 number of unique values, viewing the columns with most unique values still offer useful information on determining preprocessing parameters such as 'min_frequency' in OneHotEncoder.

In [ ]:
# number of unique values each column
nunique_counts = X_train[categorical_cols].nunique() # nunique() counts unique values

# find highest unique value count in categorical_cols
max_value = nunique_counts.max() # max() finds highest value

# In case multiple columns have same highestnunique count, get all
cols_with_max = nunique_counts[nunique_counts == max_value] # in nunique_counts, find columns with nunique count = max_value

print(cols_with_max)

When encoding categorical variables, some categories may appear very rarely in the training data. Creating a separate one-hot encoded feature for each rare category can introduce noise because the model has very little data to learn meaningful patterns from categories.

Using min_frequency=n groups categories that appear fewer than 'n' (where n is an integer) times into a single "infrequent" category/column. This reduce the number of features and helps prevent the model from overfitting to categories that appear only a handful of times.

Setting handle_unknown='infrequent_if_exist' extends this idea to the validation/test data. If the model encounters an unknown category, it is mapped to the infrequent category instead of producing an error or encoding it as all zeros ('ignore'). The intuition is that unknown categories in validation/test data are inherently rare since they weren't present in the training data. This allows the model to treat unknown categories similarly to the "infrequent" category in the training data, which often leads to more stable predictions.

In short, this approach allows the model to learn patterns for common categories individually, while grouping rare or unseen categories into a shared representation, generalizing them and reducing noise in the feature space.

In [ ]:
# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='median')

# Preprocessing for categorical data
categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(
            min_frequency=5, # categories that appear < n times (n is an integer), becomes infrequent
            handle_unknown='infrequent_if_exist' # if encoder sees category not in training set, it will map it to the "infrequent" category, if there is one
            ))
    ]
)

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

The code below tests the RandomForestRegressor model with different parameter values, specifically the max_depth, min_sample_leaf, min_samples_split, and max_features parameters

In [ ]:
# ** takes a dictionary and unpacks it i.e. {key: value} into parameters key=value.
def score_model(random_state=None, **params): # 'params' in **params is name of the dictionary, it could be anything
    """
    Score model based on RandomForestRegressor parameters passed
    """
    model = RandomForestRegressor( # default n_estimators=100
        random_state=random_state,
        n_jobs=-1, # -1 uses all available CPU cores (parallel processing), 1 or 'none' uses 1 CPU core (sequential)
        **params
    )

    pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', model)
        ]
    )
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

In [ ]:
param_grid = {
    "n_estimators": [10, 100, 1000], # number of trees
    "max_depth": [None, 10, 20], # depth limit of each tree, how tall it's allowed to split/grow
    "min_samples_leaf": [1, 2, 5], # min number of samples allowed in a leaf i.e. min_sample_leaf=5 makes model have every final prediction based at least 5 houses
    "min_samples_split": [2, 5, 10], # minimum samples needed to split a node, if can't split, node becomes leaf
    "max_features": [1.0, 'sqrt', 0.5] # max features a tree can use when splitting i.e. if dataset has 40 features, max_features='sqrt' picks out 6 random features (sqrt(40) = 6 when rounded down)
}

# for each parameter, print the score of each value
for param_name, values in param_grid.items(): #.items() returns key-value pair in tuple [key, value]
    print(f"Testing each value in {param_name} parameter:")

    for val in values: # print each score from values in parameter
        score = score_model(random_state=0,**{param_name: val}) # ** unpacks dictionary i.e. {"max_depth": 10} becomes max_depth=10
        print(f"{param_name}={val} MAE: {score}")
    
    print()

Best invidual parameter value MAE:
- n_estimators=1000 MAE: 17263.796873287672
- max_depth=None MAE: 17479.21376712329
- min_samples_leaf=1 MAE: 17479.21376712329
- min_samples_split=2 MAE: 17479.21376712329
- max_features=0.5 MAE: 17393.778287671234

We'll try combining all of these parameters into a single RandomForestRegressor model below. While the parameter values below are the best individually, keep in mind that this doesn't neccesarily correlate to a more accurate model if all of these parameters are passed into the model (this is just a good baseline on knowing where to start when tuning the model).

In [ ]:
best_individual_params = {
    "n_estimators": 1000,
    "max_depth": None,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "max_features": 0.5
}

score = score_model(random_state=0, **best_individual_params)
print(f"MAE: {score}")

The MAE using the best parameters in one model is 17153.384191780824. We can now experiment with changing each individual parameter's value to see if we can achieve an even more accurate model

In [ ]:
def average_mae(n_runs, random_state=None, **params):
    """
    Score model based on RandomForestRegressor parameters passed and average over n_runs
    """
    scores = []

    for _ in range(n_runs): # underline '_' is a throwaway variable, no use in the for loop
        mae = score_model(random_state=random_state, **params)
        scores.append(mae) # add mae to scores

    avg_mae = sum(scores) / len(scores) # sum of scores / number of runs

    return avg_mae

In [ ]:
best_params = {
    "n_estimators": 800,
    "max_depth": 10,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "max_features": 0.5
}

score = score_model(random_state=0, **best_params) # score model with random_state=0
print(f"MAE using best parameters: {score}")

average_score = average_mae(10, **best_params) # mae over 10 runs averaged
print(f"Average MAE over 10 runs: {average_score}")

The lowest MAE was with n_estimators=150, with a score of 16893.372304751952 with random_state=0. However, the sweet spot for n_estimators is 700-800, with lower MAE on average when ran multiple times without a set seed (random_state).

In [ ]:
# Define final model using best parameters
model_final = RandomForestRegressor(**best_params, n_jobs=-1, random_state=0) # n_jobs=-1 uses all available CPU cores
 
# Bundle preprocessing and modeling code in a pipeline
pipeline_final = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', model_final)
        ]
    )

# Preprocessing of training data, fit model
pipeline_final.fit(X_train, y_train)

# Preprocessing of test data, use model to make predictions
preds_test = pipeline_final.predict(X_test)

Create a csv file with id and sales predictions below

In [ ]:
# Save test predictions to file
output = pd.DataFrame(
    {
    'Id': X_test.index,
    'SalePrice': preds_test
    }
)

# uncomment the line below to create a csv file
# output.to_csv('submission.csv', index=False)  # create csv from 'output' dataframe, index=False removes index column